# LeetCode 3154 — Find Number of Ways to Reach the K-th Stair bằng Python

- Link: https://leetcode.com/problems/find-number-of-ways-to-reach-the-k-th-stair/
- Độ khó: **Hard**
- Chủ đề: **Tổ hợp** — sau `j` lần nhảy lên, vị trí là `2^j − d` với `d` = số lần xuống; các lần xuống không liền nhau ⇔ chọn `d` trong `j + 1` "khe" ⇒ đáp án `Σ_j C(j + 1, 2^j − k)`; đối chiếu bằng **DFS có nhớ** với cắt tỉa `i > k + 1`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def waysToReachStair(self, k: int) -> int:
```

## 1. Mục tiêu học

1. Mô hình hoá một "cách" là một **dãy thao tác** (có thể rỗng) kết thúc ở bậc `k`; mỗi lần đứng ở `k` sau một dãy khác nhau là một cách khác nhau.
2. Sau `j` lần lên (với `jump = 0, 1, …, j−1`), tổng quãng lên là `2⁰ + … + 2^{j−1} = 2^j − 1` ⇒ vị trí `= 1 + (2^j − 1) − d = 2^j − d`.
3. "Không xuống hai lần liên tiếp" ⇔ mỗi **khe** (trước lần lên đầu, giữa hai lần lên, sau lần lên cuối) có tối đa một lần xuống ⇒ `C(j + 1, d)` cách đặt.
4. Đáp án `Σ_{j ≥ 0} C(j + 1, 2^j − k)` (chỉ các `j` với `0 ≤ 2^j − k ≤ j + 1`) — chỉ khoảng 31 số hạng.
5. Kiểm chứng bằng DFS có nhớ trên trạng thái `(bậc, jump, vừa xuống?)`.

## 2. Đề bài & ví dụ

Cầu thang vô hạn, bậc thấp nhất là 0. Alice bắt đầu ở bậc 1 với `jump = 0`. Ở bậc `i`, một thao tác là:

- xuống bậc `i − 1` (không được dùng **hai lần liên tiếp**, và không dùng khi ở bậc 0);
- lên bậc `i + 2^jump`, rồi `jump += 1`.

Đếm số cách để tới bậc `k` (có thể tới `k` rồi thao tác tiếp và tới `k` lần nữa — tính là cách khác).

**Ví dụ 1:** `k = 0` → `2`. **Ví dụ 2:** `k = 1` → `4`.

**Ràng buộc:** `0 ≤ k ≤ 10^9`.

In [1]:
from functools import lru_cache
from math import comb
import itertools, random, time

examples = [(0, 2), (1, 4)]
for k, exp in examples:
    print('k =', k, '-> kỳ vọng', exp)

k = 0 -> kỳ vọng 2
k = 1 -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Vị trí chỉ phụ thuộc số lần lên và số lần xuống

Lần lên thứ `t` (đếm từ 0) đi `2^t` bậc. Sau `j` lần lên và `d` lần xuống (theo bất kỳ thứ tự hợp lệ nào), vị trí là `1 + (2^j − 1) − d = 2^j − d`.

### 3.2. Đếm cách xếp lần xuống không liền nhau

Một dãy gồm `j` chữ `U` và `d` chữ `D` không có hai `D` liền nhau: các `D` nằm trong `j + 1` khe quanh các `U`, mỗi khe tối đa một `D` ⇒ `C(j + 1, d)` dãy.

In [2]:
def count_no_DD(j, d):
    cnt = 0
    for pos in itertools.combinations(range(j + d), d):           # vị trí các chữ D
        if all(pos[t + 1] - pos[t] > 1 for t in range(d - 1)):
            cnt += 1
    return cnt

for j in range(0, 5):
    print(f'j = {j}:', [count_no_DD(j, d) for d in range(0, j + 3)], '| C(j+1, d):', [comb(j + 1, d) for d in range(0, j + 3)])

j = 0: [1, 1, 0] | C(j+1, d): [1, 1, 0]
j = 1: [1, 2, 1, 0] | C(j+1, d): [1, 2, 1, 0]
j = 2: [1, 3, 3, 1, 0] | C(j+1, d): [1, 3, 3, 1, 0]
j = 3: [1, 4, 6, 4, 1, 0] | C(j+1, d): [1, 4, 6, 4, 1, 0]
j = 4: [1, 5, 10, 10, 5, 1, 0] | C(j+1, d): [1, 5, 10, 10, 5, 1, 0]


### 3.3. Ràng buộc "không xuống ở bậc 0" tự động thoả

Ta chỉ đếm các dãy **kết thúc** ở `k ≥ 0`. Liệu một dãy như vậy có thể đi qua bậc âm không? Vị trí trước một lần xuống luôn `≥ 1`: bậc thấp nhất có thể là sau khi xuống từ `1` về `0`, và từ `0` không được xuống tiếp (vì xuống không liên tiếp, ngay sau xuống phải là lên). Cụ thể, sau `t` lần lên và trước lần lên tiếp theo, vị trí `≥ 2^t − (t + 1) + …` — DFS ở mục 5 kiểm chứng rằng công thức (không xét điều kiện bậc 0) khớp với mô phỏng có điều kiện.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đừng mô phỏng — **đếm theo số lần lên** `j`. Với `j` cố định, muốn kết thúc ở `k` thì cần đúng `d = 2^j − k` lần xuống, hợp lệ khi `0 ≤ d ≤ j + 1`. Vì `2^j` tăng rất nhanh, chỉ có vài `j` thoả.

Ví dụ `k = 1`: `j = 0` ⇒ `d = 0` ⇒ `C(1, 0) = 1`; `j = 1` ⇒ `d = 1` ⇒ `C(2, 1) = 2`; `j = 2` ⇒ `d = 3` ⇒ `C(3, 3) = 1`; `j = 3` ⇒ `d = 7 > 4`. Tổng `4` ✓.

## 5. Cách 1 — Brute Force: liệt kê dãy thao tác (giới hạn độ dài)

In [3]:
def brute(k, max_ups=None):
    if max_ups is None:
        max_ups = max(3, k.bit_length() + 2)
    total = 0
    stack = [(1, 0, False)]                       # (bậc, jump, vừa xuống)
    while stack:
        i, jump, down = stack.pop()
        if i == k:
            total += 1
        if not down and i > 0:
            stack.append((i - 1, jump, True))
        if jump < max_ups and i + (1 << jump) <= k + 1 + (1 << jump):   # lên (giới hạn số lần lên)
            stack.append((i + (1 << jump), jump + 1, False))
    return total

for k, exp in examples:
    print('brute:', brute(k), '| kỳ vọng', exp)

brute: 2 | kỳ vọng 2
brute: 4 | kỳ vọng 4


Brute giới hạn số lần lên `≤ bit_length(k) + 2` (sau đó `2^j − (j + 1) > k` nên không bao giờ quay về `k`) — đủ cho `k` nhỏ. Mỗi dãy được duyệt tường minh ⇒ mũ theo số thao tác.

## 6. Ý tưởng thuật toán / bước đột phá

```
ans = 0
for j in 0..31:
    d = 2^j − k
    if 0 <= d <= j + 1: ans += C(j + 1, d)
```

`O(log k)` phép tính tổ hợp nhỏ.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS có nhớ trên `(bậc, jump, vừa xuống)`

Cắt tỉa: nếu `i > k + 1` thì không bao giờ về được `k` (mỗi lần xuống chỉ 1 bậc và ngay sau phải lên ít nhất `2^jump ≥ 1`).

In [4]:
def memo_dfs(k):
    @lru_cache(maxsize=None)
    def f(i, jump, down):
        if i > k + 1:
            return 0
        res = 1 if i == k else 0
        if not down and i > 0:
            res += f(i - 1, jump, True)
        res += f(i + (1 << jump), jump + 1, False)
        return res
    return f(1, 0, False)

for k, exp in examples:
    print('DFS có nhớ:', memo_dfs(k), '| kỳ vọng', exp)

DFS có nhớ: 2 | kỳ vọng 2
DFS có nhớ: 4 | kỳ vọng 4


### 7.2. Cách 3 — Công thức tổ hợp (lời giải chính)

In [5]:
def formula(k):
    ans = 0
    for j in range(0, 32):
        d = (1 << j) - k
        if 0 <= d <= j + 1:
            ans += comb(j + 1, d)
    return ans

for k, exp in examples:
    print('công thức:', formula(k), '| kỳ vọng', exp)

công thức: 2 | kỳ vọng 2
công thức: 4 | kỳ vọng 4


### 7.3. Cách 4 — DP theo khe (không dùng `comb`)

`ways[j][d]` = số dãy `j` chữ `U`, `d` chữ `D` không có `DD` (kết thúc tuỳ ý). Đệ quy: thêm khe mới sau lần lên thứ `j`: chọn đặt 0 hoặc 1 chữ `D` ⇒ `ways[j][d] = ways[j−1][d] + ways[j−1][d−1]` với `ways[0][0] = ways[0][1] = 1` — chính là tam giác Pascal dịch một hàng.

In [6]:
def slot_dp(k):
    ways = [1, 1]                               # j = 0: khe duy nhất, 0 hoặc 1 chữ D
    ans = 0
    for j in range(0, 32):
        d = (1 << j) - k
        if 0 <= d < len(ways):
            ans += ways[d]
        ways = [ (ways[t] if t < len(ways) else 0) + (ways[t - 1] if t >= 1 else 0) for t in range(len(ways) + 1)]
    return ans

for k, exp in examples:
    print('DP theo khe:', slot_dp(k), '| kỳ vọng', exp)

DP theo khe: 2 | kỳ vọng 2
DP theo khe: 4 | kỳ vọng 4


## 8. TRACE từng bước

In các số hạng khác 0 của công thức cho vài `k`, và liệt kê tường minh các dãy thao tác cho `k = 2`.

In [7]:
for k in (0, 1, 2, 3, 6, 7, 8):
    terms = [(j, (1 << j) - k, comb(j + 1, (1 << j) - k)) for j in range(32) if 0 <= (1 << j) - k <= j + 1]
    print(f'k = {k}: (j, d, C(j+1, d)) = {terms} -> tổng {sum(t[2] for t in terms)}')

def list_ways(k, max_ups=4):
    out = []
    def go(i, jump, down, path):
        if i == k:
            out.append(''.join(path) or '(rỗng)')
        if not down and i > 0:
            go(i - 1, jump, True, path + ['D'])
        if jump < max_ups:
            go(i + (1 << jump), jump + 1, False, path + ['U'])
    go(1, 0, False, [])
    return out

print('các dãy cho k = 2:', list_ways(2))

k = 0: (j, d, C(j+1, d)) = [(0, 1, 1), (1, 2, 1)] -> tổng 2
k = 1: (j, d, C(j+1, d)) = [(0, 0, 1), (1, 1, 2), (2, 3, 1)] -> tổng 4
k = 2: (j, d, C(j+1, d)) = [(1, 0, 1), (2, 2, 3)] -> tổng 4
k = 3: (j, d, C(j+1, d)) = [(2, 1, 3)] -> tổng 3
k = 6: (j, d, C(j+1, d)) = [(3, 2, 6)] -> tổng 6
k = 7: (j, d, C(j+1, d)) = [(3, 1, 4)] -> tổng 4
k = 8: (j, d, C(j+1, d)) = [(3, 0, 1)] -> tổng 1
các dãy cho k = 2: ['DUDU', 'DUUD', 'U', 'UDUD']


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def waysToReachStair(self, k: int) -> int:
        ans = 0
        for j in range(32):
            d = (1 << j) - k
            if 0 <= d <= j + 1:
                ans += comb(j + 1, d)
        return ans

for k, exp in examples:
    got = Solution().waysToReachStair(k)
    print(got, 'OK' if got == exp else 'SAI')

2 OK
4 OK


## 10. Vì sao thuật toán đúng

1. **Song ánh:** một cách = một dãy `U/D` hợp lệ kết thúc ở `k`. Phân loại theo `j` = số `U`. Vị trí cuối `= 2^j − d` (mục 3.1) ⇒ cần `d = 2^j − k`.
2. **Đếm:** dãy `j` chữ `U`, `d` chữ `D` không có `DD` có `C(j + 1, d)` cách (mục 3.2); cần `0 ≤ d ≤ j + 1`.
3. **Điều kiện bậc 0:** không dãy nào trong số này vi phạm (kiểm chứng ở mục 12 bằng DFS mô phỏng **có** điều kiện, khớp với công thức không có điều kiện). Trực giác: trước lần lên thứ `t`, vị trí `≥ 2^t − t ≥ 1` với `t ≥ 1`, và `≥ 0` khi `t = 0`; xuống thêm tối đa một bậc trong mỗi khe nên không bao giờ dưới 0.
4. **Hữu hạn:** với `j ≥ 31`, `2^j − (j + 1) > 10^9 ≥ k` ⇒ không còn số hạng.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê dãy | mũ | `O(độ sâu)` |
| 2. DFS có nhớ | `O(log² k)` trạng thái (bậc bị cắt tỉa) | như vậy |
| 3. Công thức | `O(log k)` | `O(1)` |
| 4. DP theo khe | `O(log² k)` | `O(log k)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'memo': memo_dfs, 'formula': formula, 'slots': slot_dp, 'submit': lambda k: Solution().waysToReachStair(k)}
edge = [(2, 4), (3, 3), (4, 2), (5, 4), (6, 6), (7, 4), (8, 1), (13, 10), (15, 5), (16, 1)]
for k, exp in examples + edge:
    assert brute(k) == exp, (k, brute(k))
    for name, f in impls.items():
        assert f(k) == exp, (name, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu mọi `k ≤ 300` với DFS có nhớ (có điều kiện bậc 0)

In [10]:
for k in range(0, 301):
    m = memo_dfs(k)
    assert formula(k) == m == slot_dp(k), k
    if k <= 40:
        assert brute(k) == m
print('Công thức = DFS có nhớ = DP theo khe với mọi k ≤ 300; = liệt kê tường minh với k ≤ 40.')

Công thức = DFS có nhớ = DP theo khe với mọi k ≤ 300; = liệt kê tường minh với k ≤ 40.


### 12.3. Ca lớn (`k` tới `10^9`)

In [11]:
rng = random.Random(3154)
ks = [10 ** 9, (1 << 29), (1 << 29) - 15, (1 << 30) - 31, 999999999] + [rng.randint(0, 10 ** 9) for _ in range(5)]
t = time.time()
for k in ks:
    x = Solution().waysToReachStair(k)
    assert x == slot_dp(k)
    print(f'k = {k:10d}: {x}')
print(f'({time.time() - t:.4f}s cho {len(ks)} giá trị)')
assert Solution().waysToReachStair(1 << 29) == comb(30, 0) + 0 and Solution().waysToReachStair((1 << 30) - 31) == comb(31, 31)

k = 1000000000: 0
k =  536870912: 1
k =  536870897: 155117520
k = 1073741793: 1
k =  999999999: 0
k =  231026429: 0
k =  110874448: 0
k =  238685284: 0
k =  260550676: 0
k =   85982417: 0
(0.0009s cho 10 giá trị)


Phần lớn `k` ngẫu nhiên cho `0` — chỉ `k` rất gần một lũy thừa của 2 (trong khoảng `j + 1` bên dưới `2^j`) mới có cách.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `C(j, d)` (quên khe trước lần lên đầu hoặc sau lần lên cuối)

In [12]:
def wrong_comb(k):
    return sum(comb(j, (1 << j) - k) for j in range(32) if 0 <= (1 << j) - k <= j)   # SAI

print('sai :', wrong_comb(1), '| đúng:', Solution().waysToReachStair(1))

sai : 2 | đúng: 4


**Vì sao sai:** `j` chữ `U` tạo `j + 1` khe (kể cả hai đầu). Ví dụ `k = 1`, `j = 0`: dãy rỗng (`d = 0`) vẫn đếm; `j = 1`, `d = 1`: `DU` và `UD` — hai cách.

### Lỗi 2 — Cho phép xuống liên tiếp

In [13]:
def wrong_consecutive(k):
    @lru_cache(maxsize=None)
    def f(i, jump, depth):
        if i > k + 1 + depth or depth > 6:
            return 0
        res = 1 if i == k else 0
        if i > 0:
            res += f(i - 1, jump, depth + 1)                             # SAI: không cấm DD
        if jump < 6:
            res += f(i + (1 << jump), jump + 1, depth + 1)
        return res
    return f(1, 0, 0)

print('sai :', wrong_consecutive(0), '| đúng:', Solution().waysToReachStair(0))

sai : 7 | đúng: 2


**Vì sao sai:** dãy `U D D` (`1 → 2 → 1 → 0`) bị đếm dù có hai lần xuống liền nhau. (Hàm sai còn phải giới hạn độ sâu nhân tạo vì không có điều kiện, số cách thậm chí vô hạn khi đi lên xuống mãi.)

### Lỗi 3 — Cắt tỉa ở `i > k` thay vì `i > k + 1`

In [14]:
def wrong_prune(k):
    @lru_cache(maxsize=None)
    def f(i, jump, down):
        if i > k:                                                         # SAI: phải là k + 1
            return 0
        res = 1 if i == k else 0
        if not down and i > 0:
            res += f(i - 1, jump, True)
        res += f(i + (1 << jump), jump + 1, False)
        return res
    return f(1, 0, False)

print('sai :', wrong_prune(1), '| đúng:', Solution().waysToReachStair(1))

sai : 2 | đúng: 4


**Vì sao sai:** đứng ở `k + 1` vẫn có thể **xuống** một bậc về `k` (cách `U D` của ví dụ 2).

### Lỗi 4 — Cho phép xuống khi đang ở bậc 0 (bỏ điều kiện `i > 0`) trong DFS

In [15]:
def wrong_floor(k):
    @lru_cache(maxsize=None)
    def f(i, jump, down):
        if i > k + 1:
            return 0
        res = 1 if i == k else 0
        if not down:
            res += f(i - 1, jump, True)                                   # SAI: có thể xuống bậc -1
        res += f(i + (1 << jump), jump + 1, False)
        return res
    return f(1, 0, False)

diffs = [(k, wrong_floor(k), Solution().waysToReachStair(k)) for k in range(0, 40) if wrong_floor(k) != Solution().waysToReachStair(k)]
print('các k mà kết quả khác nhau (k, sai, đúng):', diffs[:6])

các k mà kết quả khác nhau (k, sai, đúng): []


**Kết quả:** không có `k` nào khác nhau! Trong DFS, bậc 0 chỉ đạt được ngay sau một lần **xuống** (từ 1), nên `down = True` và không thể xuống tiếp — điều kiện `i > 0` là **thừa**. Đây là "lỗi" đáng học: hiểu vì sao một điều kiện trong đề không bao giờ có hiệu lực (cũng chính là lý do công thức tổ hợp không cần xét nó). Lỗi thật sự là các lỗi 1–3.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- `k ≤ 10^9` với bước nhảy `2^jump` ⇒ chỉ `O(log k)` lần nhảy ⇒ đếm theo số lần nhảy.
- "Không dùng hai lần liên tiếp" ⇒ đặt vào **khe** giữa các thao tác khác ⇒ `C(khe, số lần)`.
- Luôn kiểm xem điều kiện biên trong đề có thực sự hạn chế gì không.

## 15. Biến thể của bài

1. Nhảy lên `b^jump` (cơ số `b ≥ 2`) ⇒ vị trí `1 + (b^j − 1)/(b − 1) − d`.
2. Cho phép xuống tối đa **hai** lần liên tiếp ⇒ mỗi khe chứa 0, 1 hoặc 2 chữ `D` — nhưng lúc này điều kiện "không xuống ở bậc 0" **có** hiệu lực (từ bậc 1 xuống hai lần sẽ thành −1), nên đếm khe thuần tuý đếm dư; dùng DFS có nhớ (ô dưới so sánh hai cách).
3. Đếm số cách dùng **đúng `m`** thao tác (bài tập 5).
4. Bắt đầu ở bậc `s` thay vì 1 ⇒ `d = s − 1 + 2^j − k`.

In [16]:
# Biến thể 1 và 2, kiểm bằng DFS có nhớ
def base_b(k, b):
    ans = 0; j = 0
    while True:
        up = (b ** j - 1) // (b - 1); d = 1 + up - k
        if d > j + 1 and up > k + 2:
            break
        if 0 <= d <= j + 1:
            ans += comb(j + 1, d)
        j += 1
    return ans

def dfs_general(k, b=2, max_run=1):
    @lru_cache(maxsize=None)
    def f(i, jump, run):
        if i > k + max_run:
            return 0
        res = 1 if i == k else 0
        if run < max_run and i > 0:
            res += f(i - 1, jump, run + 1)
        res += f(i + b ** jump, jump + 1, 0)
        return res
    return f(1, 0, 0)

for k in range(0, 60):
    for b in (3, 4):
        assert base_b(k, b) == dfs_general(k, b)
print('Biến thể cơ số b: công thức khớp DFS có nhớ với k < 60.')
# Biến thể 2: cho phép tối đa 2 lần xuống liên tiếp — điều kiện "không xuống ở bậc 0" giờ CÓ hiệu lực
naive = lambda k: sum(sum(comb(j + 1, t) * comb(j + 1 - t, (1 << j) - k - 2 * t) for t in range(0, ((1 << j) - k) // 2 + 1)
                          if 0 <= (1 << j) - k - 2 * t <= j + 1 - t) for j in range(0, 12) if (1 << j) >= k)
print('2 lần xuống liên tiếp, k = 0..5 — đếm khe (bỏ qua bậc 0):', [naive(k) for k in range(6)],
      '| DFS có điều kiện bậc 0:', [dfs_general(k, 2, 2) for k in range(6)])

Biến thể cơ số b: công thức khớp DFS có nhớ với k < 60.
2 lần xuống liên tiếp, k = 0..5 — đếm khe (bỏ qua bậc 0): [11, 14, 17, 19, 20, 16] | DFS có điều kiện bậc 0: [5, 7, 8, 9, 12, 12]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Liệt kê dãy | mũ | Oracle tường minh |
| 2 | DFS có nhớ + cắt tỉa | `O(log² k)` | Dễ nghĩ, an toàn |
| 3 | `Σ C(j + 1, 2^j − k)` | `O(log k)` | **Nộp bài** |
| 4 | DP theo khe (Pascal) | `O(log² k)` | Không cần `comb` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `k = 2`.

**Bài 2 (tính tay):** `k = 6` — liệt kê các `(j, d)` có đóng góp.

**Bài 3 (điền TODO):** liệt kê mọi dãy `U/D` cho `k` nhỏ (đúng số lượng như công thức).

**Bài 4 (tự viết lại):** biến thể 4 — bắt đầu ở bậc `s`.

**Bài 5 (mini challenge):** đếm số cách tới `k` dùng **đúng `m`** thao tác.

In [17]:
def list_sequences_todo(k):
    # TODO
    raise NotImplementedError

def start_at_s_todo(k, s):
    # TODO
    raise NotImplementedError

def exactly_m_ops_todo(k, m):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `j = 1`: `d = 0` ⇒ `C(2, 0) = 1`. (2) `j = 2`: `d = 2` ⇒ `C(3, 2) = 3`. (3) `4`.

**Bài 2.** (1) `j = 3`: `d = 2` ⇒ `C(4, 2) = 6`. (2) `j = 4`: `d = 10 > 5`. (3) `6`.

**Bài 3.** (1) Chọn `j`, `d = 2^j − k`. (2) Chọn `d` khe trong `j + 1` khe. (3) Ghép chuỗi.

**Bài 4.** (1) Vị trí `= s + 2^j − 1 − d`. (2) `d = s − 1 + 2^j − k`. (3) Kiểm DFS bắt đầu từ `s ≥ 1` (với `s = 0` thì khe đầu tiên không được chứa `D` — điều kiện bậc 0 có hiệu lực, công thức phải đổi thành `C(j, d)`).

**Bài 5.** (1) `m = j + d`. (2) Với mỗi `j`, `d = 2^j − k` cố định ⇒ chỉ nhận khi `j + d = m`. (3) Kiểm DFS có thêm chiều "số thao tác".

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().waysToReachStair(2) == 4 and Solution().waysToReachStair(6) == 6

def list_sequences(k):
    out = []
    for j in range(32):
        d = (1 << j) - k
        if 0 <= d <= j + 1:
            for slots in itertools.combinations(range(j + 1), d):
                s = ''
                for g in range(j + 1):
                    if g in slots:
                        s += 'D'
                    if g < j:
                        s += 'U'
                out.append(s)
    return out

def simulate(seq):
    i, jump, prev = 1, 0, ''
    for c in seq:
        if c == 'D':
            assert prev != 'D' and i > 0; i -= 1
        else:
            i += 1 << jump; jump += 1
        prev = c
    return i

for k in range(0, 40):
    seqs = list_sequences(k)
    assert len(seqs) == len(set(seqs)) == Solution().waysToReachStair(k)
    assert all(simulate(s) == k for s in seqs)
print('Bài 1: 4 | Bài 2: 6 | Bài 3: k = 2 ->', list_sequences(2))

Bài 1: 4 | Bài 2: 6 | Bài 3: k = 2 -> ['U', 'DUDU', 'DUUD', 'UDUD']


In [19]:
# Bài 4, 5
def start_at_s(k, s):
    ans = 0
    for j in range(40):
        d = s - 1 + (1 << j) - k
        if 0 <= d <= j + 1:
            ans += comb(j + 1, d)
    return ans

def exactly_m_ops(k, m):
    ans = 0
    for j in range(40):
        d = (1 << j) - k
        if 0 <= d <= j + 1 and j + d == m:
            ans += comb(j + 1, d)
    return ans

def dfs_start(k, s):
    @lru_cache(maxsize=None)
    def f(i, jump, down):
        if i > k + 1:
            return 0
        res = 1 if i == k else 0
        if not down and i > 0:
            res += f(i - 1, jump, True)
        res += f(i + (1 << jump), jump + 1, False)
        return res
    return f(s, 0, False)

def dfs_m(k, m):
    @lru_cache(maxsize=None)
    def f(i, jump, down, left):
        if left == 0:
            return 1 if i == k else 0
        if i > k + left:
            return 0
        res = 0
        if not down and i > 0:
            res += f(i - 1, jump, True, left - 1)
        res += f(i + (1 << jump), jump + 1, False, left - 1)
        return res
    return f(1, 0, False, m)

for k in range(0, 40):
    for s in range(1, 6):                              # s = 0: điều kiện bậc 0 có hiệu lực (không xuống được ngay)
        if k + 1 >= s:
            assert start_at_s(k, s) == dfs_start(k, s), (k, s)
    for m in range(0, 12):
        assert exactly_m_ops(k, m) == dfs_m(k, m)
print('Bài 4 (bắt đầu ở s) và Bài 5 (đúng m thao tác) khớp DFS.')

Bài 4 (bắt đầu ở s) và Bài 5 (đúng m thao tác) khớp DFS.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ans = 0
for j in range(32):
    d = (1 << j) - k              # số lần xuống cần có
    if 0 <= d <= j + 1:
        ans += comb(j + 1, d)     # đặt d lần xuống vào j + 1 khe
```

**Checklist:**

- [ ] Vị trí sau `j` lần lên, `d` lần xuống là `2^j − d` (bắt đầu từ bậc 1)?
- [ ] `j + 1` khe (không phải `j`)?
- [ ] Đủ `j` tới ~31 cho `k ≤ 10^9`?
- [ ] Dãy rỗng (`k = 1`, `j = 0`) được đếm?